# ITBS-VoGen — Colab Training

Google Colab (T4 GPU) で RVC 話者モデルを学習するためのノートブック。ローカル (Mac) では推論のみ行い、重い学習だけクラウドに逃がす運用。

本ノートブックは **複数データセットから複数モデルを一度に学習する**構成。`SPEAKER_CONFIGS` に学習対象を列挙すると、依存セットアップは1回だけ走り、学習・保存は各モデル分ループする。

## 事前準備

1. **Google Drive に学習データをアップロード**: `MyDrive/itbs-vogen/Train/` 以下に各データセットフォルダ（例: `set1-1/`, `set1-2/`, `set1-3/`）を配置する。
2. **ランタイム変更**: メニュー → ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータ = **T4 GPU**。
3. 以下のセルを上から順に実行。

## 所要時間の目安

T4 で 1 モデルあたり 20〜40 分（29分データ / 200 epoch 前提）。3モデルなら **合計1.5〜2時間**。Colab 無料枠のセッションタイムアウトに注意。途中で切れた場合は `SPEAKER_CONFIGS` から完了済みを消して再開すれば中断点から進められる。


## 1. GPU 確認

In [ ]:
!nvidia-smi

## 2. Google Drive マウント

学習データ読み込みと成果物保存に使用。認証プロンプトが出るので承認する。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 学習設定

学習対象のデータセットを `SPEAKER_CONFIGS` に列挙する。各エントリは `(モデル名, Drive上のデータディレクトリ名)`。

`DRIVE_ROOT` 配下の想定構造:
```
itbs-vogen/
├── Train/
│   ├── set1-1/*.wav
│   ├── set1-2/*.wav
│   └── set1-3/*.wav
└── models/            ← 各モデルの .pth / .index がここに吐かれる
```


In [ ]:
import os
from pathlib import Path

# === 学習対象 ===
# (モデル名, Drive 上のデータディレクトリ名)
SPEAKER_CONFIGS = [
    ('itbs_set1_1', 'set1-1'),
    ('itbs_set1_2', 'set1-2'),
    ('itbs_set1_3', 'set1-3'),
]

# === 共通ハイパーパラメータ ===
TOTAL_EPOCHS = 200                 # 典型値: 150-300
SAVE_EVERY_EPOCH = 50              # 途中保存の粒度
BATCH_SIZE = 12                    # T4 なら 12、A100 なら 24+
SR = '48k'                         # 32k / 40k / 48k
VERSION = 'v2'

# === Drive 上のパス ===
DRIVE_ROOT = Path('/content/drive/MyDrive/itbs-vogen')
DRIVE_TRAIN_ROOT = DRIVE_ROOT / 'Train'
DRIVE_MODELS_ROOT = DRIVE_ROOT / 'models'

# === Colab 作業ディレクトリ ===
WORK_DIR = Path('/content/ITBS-VoGen')

# 事前チェック
DRIVE_MODELS_ROOT.mkdir(parents=True, exist_ok=True)
for name, subdir in SPEAKER_CONFIGS:
    src = DRIVE_TRAIN_ROOT / subdir
    assert src.exists(), f'Training data not found: {src}'
    wavs = list(src.glob('*.wav'))
    print(f'[{name}] {src} -> {len(wavs)} wav files')


## 4. リポジトリ取得

公開リポジトリとRVC submoduleを取得。

In [ ]:
%cd /content
![ -d /content/ITBS-VoGen ] || git clone --recurse-submodules https://github.com/Arimuri/ITBS-VoGen.git
%cd /content/ITBS-VoGen
!git pull
!git submodule update --init --recursive

## 5. Python 3.10 + 依存インストール

Colab のデフォルト Python は 3.12 だが、RVC のピン (`numba==0.56.4`, `fairseq==0.12.2`) が 3.12 非対応なので **Python 3.10 を apt で追加**し、以降は `python3.10 -m pip` で明示的に使う。

5-10分かかる。途中で「Gemini が pyproject.toml を直しましょうか?」という提案が出ても **拒否**すること（こちらで設計済みの制約を知らないので、勝手に依存を足すと RVC と衝突する）。万一改変されたら `!git checkout pyproject.toml` で戻せる。


In [ ]:
%cd /content/ITBS-VoGen

# Defensive reset: if anything (Gemini, a failed edit, a stale state) modified
# our pinned project metadata, revert it before installing.
!git checkout pyproject.toml

# ---- Python 3.10 -----------------------------------------------------------
# Colab defaults have moved to 3.12; RVC pins (numba 0.56.4, fairseq 0.12.2) require 3.10.
# Try the distro package first; fall back to the deadsnakes PPA if not available.
!apt-get update -q
!apt-get install -y -q python3.10 python3.10-dev python3.10-venv python3.10-distutils \
  || ( apt-get install -y -q software-properties-common \
       && add-apt-repository -y ppa:deadsnakes/ppa \
       && apt-get update -q \
       && apt-get install -y -q python3.10 python3.10-dev python3.10-venv python3.10-distutils )
!python3.10 --version

# Install pip for the freshly-added Python 3.10.
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py
!python3.10 /tmp/get-pip.py --quiet
!python3.10 -m pip --version

# ---- Dependencies ----------------------------------------------------------
# RVC ships an old omegaconf whose metadata trips pip >= 24.1.
!python3.10 -m pip install --quiet 'pip<24.1'

# RVC's deps + our package (all inside python3.10).
# torch version is not pinned; the fairseq/torch.load weights_only issue is
# handled at runtime by src/itbs_vogen/_compat/sitecustomize.py (injected
# via PYTHONPATH when the wrapper spawns RVC subprocesses).
!python3.10 -m pip install --quiet -r third_party/rvc/requirements.txt
!python3.10 -m pip install --quiet -e .

# ---- Verify ----------------------------------------------------------------
!python3.10 -c "import sys, torch, fairseq, faiss; \
  print('py:', sys.version.split()[0]); \
  print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available()); \
  print('fairseq:', fairseq.__version__); print('faiss ok')"
!python3.10 -m itbs_vogen.cli --help


## 6. ベースモデル取得

HuBERT + RMVPE + pretrained_v2 (48k, F0あり) をDL。初回のみ。

In [ ]:
!bash scripts/download_models.sh


## 7. 学習データ配置

Drive の各データセットを Colab ローカル (`/content/ITBS-VoGen/Train/<dir>/`) にコピーする (Drive直読みは遅い)。


In [ ]:
import shutil

for _, subdir in SPEAKER_CONFIGS:
    src = DRIVE_TRAIN_ROOT / subdir
    dst = WORK_DIR / 'Train' / subdir
    dst.mkdir(parents=True, exist_ok=True)
    for wav in src.glob('*.wav'):
        target = dst / wav.name
        if not target.exists():
            shutil.copy(wav, target)
    print(f'[{subdir}] copied {len(list(dst.glob("*.wav")))} wavs to {dst}')


## 8. プリフライトチェック

学習開始前に全ての前提条件が揃っているか確認。赤字が出たら該当するセルに戻って再実行すること。


In [ ]:
import sys
from pathlib import Path

problems = []

# Pretrained base models (downloaded by scripts/download_models.sh).
required = [
    WORK_DIR / 'third_party/rvc/assets/hubert/hubert_base.pt',
    WORK_DIR / 'third_party/rvc/assets/rmvpe/rmvpe.pt',
    WORK_DIR / f'third_party/rvc/assets/pretrained_v2/f0G{SR}.pth',
    WORK_DIR / f'third_party/rvc/assets/pretrained_v2/f0D{SR}.pth',
]
for p in required:
    if not p.exists():
        problems.append(f'MISSING pretrained: {p}  (re-run "6. ベースモデル取得")')
    else:
        print(f'ok  pretrained {p.name}  ({p.stat().st_size/1e6:.1f}MB)')

# Training data must be copied into the container.
for name, subdir in SPEAKER_CONFIGS:
    local = WORK_DIR / 'Train' / subdir
    wavs = list(local.glob('*.wav')) if local.exists() else []
    if not wavs:
        problems.append(f'MISSING training data: {local}  (re-run "7. 学習データ配置")')
    else:
        print(f'ok  {name}: {len(wavs)} wavs in {local}')

# Python 3.10 + imports must all work.
import subprocess as sp
r = sp.run(['python3.10', '-c',
    'import torch; assert torch.cuda.is_available(), "CUDA not available!"'],
    capture_output=True, text=True)
if r.returncode != 0:
    problems.append(f'Python 3.10 CUDA check failed:\n{r.stdout}\n{r.stderr}')
else:
    print('ok  python3.10 + CUDA')

if problems:
    print('\n=== PROBLEMS FOUND ===')
    for p in problems:
        print('  - ' + p)
    raise RuntimeError(f'{len(problems)} precondition(s) failed. Fix and rerun this cell.')
print('\nAll preconditions met. Proceed to training.')


## 9. 学習実行 (ループ)

`SPEAKER_CONFIGS` の全エントリを順番に学習する。T4 で各モデル 20-40 分、合計で数時間。サブプロセスの出力（RVC の各ステージ: preprocess, f0, feature, train, index）がリアルタイムで流れる。

各モデル完成直後に Drive に保存されるので、途中でセッションが切れても完了済みは残る。


In [ ]:
import subprocess
import time

for name, subdir in SPEAKER_CONFIGS:
    local_train = WORK_DIR / 'Train' / subdir
    drive_out = DRIVE_MODELS_ROOT / name
    drive_out.mkdir(parents=True, exist_ok=True)

    if (drive_out / 'model.pth').exists() and (drive_out / 'model.index').exists():
        print(f'[{name}] already trained (found in Drive). Skipping.')
        continue

    print(f'\n===== training {name} from {local_train} =====')
    t0 = time.time()
    cmd = [
        'python3.10', '-u', '-m', 'itbs_vogen.cli', 'train',
        '-s', name,
        '-d', str(local_train),
        '--sr', SR,
        '--version', VERSION,
        '--epochs', str(TOTAL_EPOCHS),
        '--save-every', str(SAVE_EVERY_EPOCH),
        '--batch-size', str(BATCH_SIZE),
        '--n-proc', '4',
        '--device', 'cuda:0',
    ]
    # Stream output live and raise with context if the subprocess fails.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Training for {name} failed with exit {proc.returncode}. See output above.')

    local_model = WORK_DIR / 'models' / 'speakers' / name
    for fname in ('model.pth', 'model.index'):
        src = local_model / fname
        if src.exists():
            shutil.copy(src, drive_out / fname)
    elapsed = time.time() - t0
    print(f'[{name}] done in {elapsed/60:.1f} min, saved to {drive_out}')


## 10. 保存確認

全モデルが Drive に保存されているか確認。


In [ ]:
for name, _ in SPEAKER_CONFIGS:
    drive_out = DRIVE_MODELS_ROOT / name
    pth = drive_out / 'model.pth'
    idx = drive_out / 'model.index'
    pth_ok = pth.exists()
    idx_ok = idx.exists()
    pth_size = f'{pth.stat().st_size / 1e6:.1f}MB' if pth_ok else 'missing'
    idx_size = f'{idx.stat().st_size / 1e6:.1f}MB' if idx_ok else 'missing'
    print(f'[{name}] pth: {pth_size}  |  index: {idx_size}')


## 11. 推論 (Colab)

Mac の MPS / CPU 組み合わせは RVC のデコーダで未サポート ops に当たるため、推論も Colab 側で実行する。手持ちの変換対象ファイルを Drive にアップロードして、学習済み 3 モデル全てで変換→結果を Drive に保存する。

**事前準備**: Drive の `MyDrive/itbs-vogen/inputs/` に変換したい wav ファイルをアップロードする。


In [ ]:
import shutil
import subprocess
from pathlib import Path

# === 推論設定 ===
# Drive 上の入力ファイル (事前にアップロードしておく)
# Apollo 前処理済みを使うときは 'test_HonestyVo_apollo.wav' に差し替える
INFER_INPUT = DRIVE_ROOT / 'inputs' / 'test_HonestyVo.wav'

# 推論パラメータ
INFER_F0_METHOD = 'rmvpe'
INFER_F0_UP_KEY = 0       # メロディ保持なら 0
INFER_INDEX_RATE = 0.5    # 話者音色の反映度 (0.3-0.75 で探る)
INFER_PROTECT = 0.5       # 子音・息成分保護 (0.33-0.5)
INFER_RMS_MIX_RATE = 0.25 # 音量エンベロープ追従率 (0.25 = RVC本家推奨)

# === 実行 ===
assert INFER_INPUT.exists(), f'Input not found: {INFER_INPUT}. Upload wav to Drive first.'
print(f'input: {INFER_INPUT}  ({INFER_INPUT.stat().st_size/1e6:.1f}MB)')

DRIVE_INFER_OUT = DRIVE_ROOT / 'outputs'
DRIVE_INFER_OUT.mkdir(parents=True, exist_ok=True)

# Mirror Drive-side trained models into local models/speakers/<name>/ so the
# wrapper can load them with its standard speaker-dir convention.
local_models_root = WORK_DIR / 'models' / 'speakers'
for name, _ in SPEAKER_CONFIGS:
    src_dir = DRIVE_MODELS_ROOT / name
    dst_dir = local_models_root / name
    dst_dir.mkdir(parents=True, exist_ok=True)
    for fname in ('model.pth', 'model.index'):
        src, dst = src_dir / fname, dst_dir / fname
        if src.exists() and not dst.exists():
            shutil.copy(src, dst)

# Copy input to a local path (Drive reads can be slow).
local_input = WORK_DIR / 'inputs' / INFER_INPUT.name
local_input.parent.mkdir(parents=True, exist_ok=True)
if not local_input.exists():
    shutil.copy(INFER_INPUT, local_input)

# Run inference per speaker model.
for name, _ in SPEAKER_CONFIGS:
    stem = local_input.stem
    out = WORK_DIR / 'outputs' / f'{stem}_{name}.wav'
    out.parent.mkdir(parents=True, exist_ok=True)
    print(f'\n===== inferring {name} -> {out.name} =====')

    cmd = [
        'python3.10', '-u', '-m', 'itbs_vogen.cli', 'infer',
        str(local_input),
        '-o', str(out),
        '-s', name,
        '--f0-method', INFER_F0_METHOD,
        '--f0-up-key', str(INFER_F0_UP_KEY),
        '--index-rate', str(INFER_INDEX_RATE),
        '--protect', str(INFER_PROTECT),
        '--rms-mix-rate', str(INFER_RMS_MIX_RATE),
        '--device', 'cuda:0',
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()

    if proc.returncode == 0 and out.exists():
        drive_out = DRIVE_INFER_OUT / out.name
        shutil.copy(out, drive_out)
        print(f'[{name}] saved: {drive_out}  ({drive_out.stat().st_size/1e6:.1f}MB)')
    else:
        print(f'[{name}] FAILED (exit {proc.returncode})')


In [ ]:
import shutil
import subprocess
from pathlib import Path

APOLLO_DIR = Path('/content/Apollo')
APOLLO_CKPT = APOLLO_DIR / 'model' / 'apollo_vocal.ckpt'
APOLLO_CFG = APOLLO_DIR / 'model' / 'apollo_vocal.yaml'

APOLLO_INPUT = WORK_DIR / 'inputs' / 'test_HonestyVo.wav'
APOLLO_OUTPUT = WORK_DIR / 'inputs' / 'test_HonestyVo_apollo.wav'

# --- 1. Apollo 本体を取得 --------------------------------------------------
if not APOLLO_DIR.exists():
    !cd /content && git clone https://github.com/JusperLee/Apollo.git
else:
    print(f'{APOLLO_DIR} exists, reusing.')

# --- 2. Apollo の依存 ------------------------------------------------------
!python3.10 -m pip install -q omegaconf ml_collections pytorch-lightning auraloss torchmetrics einops
!cd /content/Apollo && python3.10 -m pip install -q -e . 2>/dev/null || true

# --- 3. ckpt + config DL (baicai1145 vocal-msst) --------------------------
APOLLO_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not APOLLO_CKPT.exists():
    !curl -L --fail -o {APOLLO_CKPT} \
      "https://huggingface.co/baicai1145/Apollo-vocal-msst/resolve/main/model_apollo_vocals_ep_54.ckpt"
if not APOLLO_CFG.exists():
    !curl -L --fail -o {APOLLO_CFG} \
      "https://huggingface.co/baicai1145/Apollo-vocal-msst/resolve/main/config_apollo_vocals_ep_54.yaml"

assert APOLLO_INPUT.exists(), f'Apollo input not found: {APOLLO_INPUT}'

# --- 4. チャンク分割 + クロスフェードで推論 -------------------------------
# T4 (14GiB) では 29秒の 44.1kHz ステレオを丸ごと流すと OOM する (~5GiB の中間
# テンソル)。10秒チャンク + 2秒オーバーラップで流して、オーバーラップ区間を
# 線形クロスフェードで繋ぐ。
apollo_script = '''
import os, sys, yaml, torch, numpy as np, soundfile as sf, librosa
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/Apollo")
from look2hear.models.apollo import Apollo

IN_WAV = "%s"
OUT_WAV = "%s"
CKPT = "%s"
CFG = "%s"
CHUNK_SEC = 10.0
OVERLAP_SEC = 2.0

with open(CFG) as f:
    cfg = yaml.safe_load(f) or {}
audio_cfg = cfg.get("audio", {}) or {}
model_cfg = cfg.get("model", {}) or {}
arch = {}
for k in ("sr", "win", "feature_dim", "layer"):
    if k in audio_cfg: arch[k] = audio_cfg[k]
    elif k in model_cfg: arch[k] = model_cfg[k]
arch.setdefault("sr", 44100); arch.setdefault("win", 20)
arch.setdefault("feature_dim", 384); arch.setdefault("layer", 8)
print("arch:", arch)

model = Apollo(**arch)
state = torch.load(CKPT, map_location="cpu", weights_only=False)
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]
cleaned = {}
for k, v in state.items():
    nk = k
    for prefix in ("model.", "module."):
        if nk.startswith(prefix):
            nk = nk[len(prefix):]; break
    cleaned[nk] = v
missing, unexpected = model.load_state_dict(cleaned, strict=False)
print("state_dict  missing:", len(missing), "unexpected:", len(unexpected))
model = model.cuda().eval()

audio, _ = librosa.load(IN_WAV, sr=arch["sr"], mono=False)
if audio.ndim == 1:
    audio = np.stack([audio, audio], axis=0)
C, T = audio.shape
print(f"audio: ch={C}, samples={T}, duration={T/arch['sr']:.1f}s")

chunk = int(CHUNK_SEC * arch["sr"])
hop   = int((CHUNK_SEC - OVERLAP_SEC) * arch["sr"])
ov    = int(OVERLAP_SEC * arch["sr"])

out = np.zeros((C, T), dtype=np.float32)
weight = np.zeros(T, dtype=np.float32)
fade_in  = np.linspace(0.0, 1.0, ov, dtype=np.float32)
fade_out = np.linspace(1.0, 0.0, ov, dtype=np.float32)

pos = 0; idx = 0
while pos < T:
    end = min(pos + chunk, T)
    x = torch.from_numpy(audio[:, pos:end]).float().unsqueeze(0).cuda()
    with torch.no_grad():
        y = model(x)
    y = y.squeeze(0).cpu().numpy()  # (C, L)
    L = y.shape[-1]

    w = np.ones(L, dtype=np.float32)
    if pos > 0 and L >= ov:         # fade in beginning (except first chunk)
        w[:ov] = fade_in
    if end < T and L >= ov:         # fade out end (except last chunk)
        w[-ov:] = fade_out

    out[:, pos:pos+L] += y * w
    weight[pos:pos+L] += w
    idx += 1
    print(f"  chunk {idx}: [{pos/arch['sr']:.1f}-{end/arch['sr']:.1f}]s")
    torch.cuda.empty_cache()
    if end >= T: break
    pos += hop

out /= np.maximum(weight, 1e-8)
# write as (T, C)
sf.write(OUT_WAV, out.T, arch["sr"])
print("wrote:", OUT_WAV)
''' % (str(APOLLO_INPUT), str(APOLLO_OUTPUT), str(APOLLO_CKPT), str(APOLLO_CFG))

script_path = Path('/content/apollo_infer.py')
script_path.write_text(apollo_script)
!python3.10 {script_path}

# --- 5. Drive にコピー ----------------------------------------------------
if APOLLO_OUTPUT.exists():
    drive_out = DRIVE_ROOT / 'inputs' / APOLLO_OUTPUT.name
    shutil.copy(APOLLO_OUTPUT, drive_out)
    print(f'\nSaved: {APOLLO_OUTPUT}  ({APOLLO_OUTPUT.stat().st_size/1e6:.1f}MB)')
    print(f'Saved to Drive: {drive_out}')
else:
    print('\nApollo inference failed. See errors above.')


In [ ]:
import shutil
import subprocess
from pathlib import Path

APOLLO_DIR = Path('/content/Apollo')
APOLLO_CKPT = APOLLO_DIR / 'model' / 'apollo_vocal.ckpt'
APOLLO_CFG = APOLLO_DIR / 'model' / 'apollo_vocal.yaml'

APOLLO_INPUT = WORK_DIR / 'inputs' / 'test_HonestyVo.wav'  # 既に「11. 推論」のセットアップでコピー済み想定
APOLLO_OUTPUT = WORK_DIR / 'inputs' / 'test_HonestyVo_apollo.wav'

# --- 1. Apollo 本体を取得 (look2hear パッケージ用) -------------------------
if not APOLLO_DIR.exists():
    !cd /content && git clone https://github.com/JusperLee/Apollo.git
else:
    print(f'{APOLLO_DIR} exists, reusing.')

# --- 2. Apollo の依存 (conda を避けて pip で) ------------------------------
!python3.10 -m pip install -q omegaconf ml_collections pytorch-lightning auraloss torchmetrics einops

# --- 3. look2hear を python3.10 から import 可能にする ---------------------
!cd /content/Apollo && python3.10 -m pip install -q -e . 2>/dev/null || true

# --- 4. 事前学習済みチェックポイントDL (baicai1145 の vocal-msst) ---------
APOLLO_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not APOLLO_CKPT.exists():
    !curl -L --fail -o {APOLLO_CKPT} \
      "https://huggingface.co/baicai1145/Apollo-vocal-msst/resolve/main/model_apollo_vocals_ep_54.ckpt"
if not APOLLO_CFG.exists():
    !curl -L --fail -o {APOLLO_CFG} \
      "https://huggingface.co/baicai1145/Apollo-vocal-msst/resolve/main/config_apollo_vocals_ep_54.yaml"
!ls -la {APOLLO_CKPT.parent}

# --- 5. 入力ファイルの存在確認 ---------------------------------------------
assert APOLLO_INPUT.exists(), (
    f'Apollo input not found: {APOLLO_INPUT}. '
    f'「11. 推論」セルを先に一度実行して Drive から /content/ITBS-VoGen/inputs/ '
    f'にコピーしておくか、手動でアップロードすること。'
)

# --- 6. Apollo 推論スクリプト (look2hear + config YAML ベース) -------------
apollo_script = '''
import sys, yaml, torch, numpy as np, soundfile as sf, librosa
sys.path.insert(0, "/content/Apollo")
import look2hear.models

IN_WAV = "%s"
OUT_WAV = "%s"
CKPT = "%s"
CFG = "%s"

with open(CFG) as f:
    cfg = yaml.safe_load(f) or {}

# Config の audio パラメータで look2hear モデル生成
audio_cfg = cfg.get("audio", {})
model_cfg = cfg.get("model", {})

kwargs = {}
for k in ("sr", "win", "feature_dim", "layer"):
    if k in audio_cfg:
        kwargs[k] = audio_cfg[k]
    elif k in model_cfg:
        kwargs[k] = model_cfg[k]
# フォールバック値 (Lew/baicai1145 モデルの標準)
kwargs.setdefault("sr", 44100)
kwargs.setdefault("win", 20)
kwargs.setdefault("feature_dim", 256)
kwargs.setdefault("layer", 6)
print("model kwargs:", kwargs)

model = look2hear.models.BaseModel.from_pretrain(CKPT, **kwargs).cuda().eval()

# load audio (Apollo は 44.1kHz ステレオ想定)
audio, sr = librosa.load(IN_WAV, sr=kwargs["sr"], mono=False)
if audio.ndim == 1:
    audio = np.stack([audio, audio], axis=0)
x = torch.from_numpy(audio).float().unsqueeze(0).cuda()  # (1, C, T)
print("input shape:", x.shape)

with torch.no_grad():
    y = model(x)  # 期待 shape (1, C, T)
if y.ndim == 3:
    y = y.squeeze(0).cpu().numpy()
else:
    y = y.cpu().numpy()

# ステレオ -> (T, 2) for soundfile
if y.ndim == 2 and y.shape[0] == 2:
    y = y.T
sf.write(OUT_WAV, y, kwargs["sr"])
print("wrote:", OUT_WAV)
''' % (str(APOLLO_INPUT), str(APOLLO_OUTPUT), str(APOLLO_CKPT), str(APOLLO_CFG))

script_path = Path('/content/apollo_infer.py')
script_path.write_text(apollo_script)

!python3.10 {script_path}

# --- 7. 結果を Drive にも保存 --------------------------------------------
if APOLLO_OUTPUT.exists():
    drive_out = DRIVE_ROOT / 'inputs' / APOLLO_OUTPUT.name
    shutil.copy(APOLLO_OUTPUT, drive_out)
    print(f'\nSaved: {APOLLO_OUTPUT}  ({APOLLO_OUTPUT.stat().st_size/1e6:.1f}MB)')
    print(f'Saved to Drive: {drive_out}')
else:
    print('\nApollo inference failed. See errors above.')


## 12. ローカル (Mac) で結果を使う

Drive の `MyDrive/itbs-vogen/outputs/` に以下のファイルが出来ているはず:

```
test_HonestyVo_itbs_set1_1.wav
test_HonestyVo_itbs_set1_2.wav
test_HonestyVo_itbs_set1_3.wav
```

Mac にダウンロードして聴き比べる。配置例:

```
/Users/arimura/Documents/dev/ITBS_VoGen/outputs/
├── test_HonestyVo_itbs_set1_1.wav
├── test_HonestyVo_itbs_set1_2.wav
└── test_HonestyVo_itbs_set1_3.wav
```

## パラメータ調整

推論品質が不満なら「11. 推論」セルの以下を変えて再実行:

- `INFER_INDEX_RATE` (0.0〜1.0): 高いほど学習した話者音色に寄る。元の声の癖を消したいなら 0.75〜0.9、元の情報を残したいなら 0.4〜0.5
- `INFER_PROTECT` (0.0〜0.5): 子音保護。子音がモゴつくなら 0.5 に近づける
- `INFER_F0_UP_KEY` (半音): メロディをそのまま保つなら 0、キーを上下させたいならここで
- `INFER_F0_METHOD`: `rmvpe` 固定推奨。劣化入力に最も強い

## トラブルシュート

- **OOM**: 長い入力ファイルだと CUDA メモリが足りない場合がある。入力を分割するか `--batch-size` 相当のパラメータを下げる
- **ノイズが目立つ**: 学習のエポック数が足りない可能性。エポックを 300-500 に増やして再学習する
- **元の劣化ガビガビが残る**: 劣化が深すぎる場合、RVC だけでは消しきれないことも。前処理で denoise を挟むか、`INFER_INDEX_RATE` を高くする
